In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
import sys
import math
import logging
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
from pixell import reproject, lensing, enmap, utils

sys.path.append("/users/stevensonb/Research/tools/deepsphere-cosmo-tf2")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf

tf.get_logger().setLevel(logging.ERROR)

from tensorflow.keras import layers
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Flatten,
    LeakyReLU,
    Add,
    BatchNormalization,
)
from tensorflow.keras.callbacks import EarlyStopping, TerminateOnNaN, TensorBoard
from tensorflow.keras.optimizers import AdamW, Adam
from tensorflow.keras.optimizers.schedules import CosineDecayRestarts, ExponentialDecay

from deepsphere import HealpyGCNN
from deepsphere.healpy_layers import (
    HealpyChebyshev,
    HealpyPool,
    Healpy_ResidualLayer,
    HealpyPseudoConv_Transpose,
)

from mlpng import Core
from mlpng.utils import (
    setup_logging,
    try_init_wandb,
    get_data,
    make_trainer_plots,
    PowerSpectrumLoss,
    PowerSpectrumLoss2,
    RMSELoss,
    RMSELoss2,
    rmse_metrics,
)
from mlpng.utils.dataloaders import MapDataset, UnlensMapDataset, PhiMapDataset

from mlpng.scn_jorik import get_model as get_fnl_model

logger = setup_logging("mlpng.notebook", level=logging.DEBUG)

In [ ]:
print("Conda environment:", os.environ.get("CONDA_DEFAULT_ENV", "Not set"))
print("Python executable:", sys.executable)
print(f"TensorFlow version: {tf.__version__}")

In [ ]:
@tf.keras.saving.register_keras_serializable()
class EncoderBlock(tf.keras.layers.Layer):
    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        activation,
        max_batch_size,
        K,
        use_bn=True,
        use_bias=False,
        pool=True,
        dropout_rate=0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.nside = nside
        self.npix = npix
        self.fin = fin
        self.fout = fout
        self.activation_fn = activation
        self.max_batch_size = max_batch_size
        self.K = K
        self.use_bn = use_bn
        self.use_bias = use_bias
        self.dropout_rate = dropout_rate
        self.pool = pool
        indices = np.arange(npix)

        enc_layers = [
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation=activation,
                use_bn=use_bn,
                use_bias=use_bias,
            ),
            HealpyChebyshev(
                K=K,
                Fout=fout,
                activation=activation,
                use_bn=use_bn,
                use_bias=use_bias,
            ),
        ]
        if dropout_rate > 0.0:
            enc_layers.append(Dropout(dropout_rate))

        self.body = HealpyGCNN(
            nside=nside,
            indices=indices,
            layers=enc_layers,
            n_neighbors=8,
            max_batch_size=max_batch_size,
            initial_Fin=fin,
        )

        self.pooler = None
        if pool:
            self.pooler = HealpyGCNN(
                nside=nside,
                indices=indices,
                layers=[HealpyPool(1, "AVG")],
                n_neighbors=8,
                max_batch_size=max_batch_size,
                initial_Fin=fin,
            )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "nside": self.nside,
                "npix": self.npix,
                "fin": self.fin,
                "fout": self.fout,
                "activation": self.activation_fn,
                "max_batch_size": self.max_batch_size,
                "K": self.K,
                "use_bn": self.use_bn,
                "use_bias": self.use_bias,
                "pool": self.pool,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config

    def call(self, inputs, training=False):
        x = skip = self.body(inputs, training=training)
        if self.pool:
            x = self.pooler(x, training=training)
        return x, skip


@tf.keras.saving.register_keras_serializable()
class DecoderBlock(tf.keras.layers.Layer):
    def __init__(
        self,
        nside,
        npix,
        fin,
        fout,
        activation,
        K,
        max_batch_size,
        upsample=True,
        use_bn=True,
        use_bias=False,
        dropout_rate=0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.nside = nside
        self.npix = npix
        self.fin = fin
        self.fout = fout
        self.activation_fn = activation
        self.max_batch_size = max_batch_size
        self.K = K
        self.use_bn = use_bn
        self.use_bias = use_bias
        self.dropout_rate = dropout_rate
        self.upsample = upsample

        indices = np.arange(npix)

        dec_layers = []
        if upsample:
            dec_layers.append(HealpyPseudoConv_Transpose(1, fout))
        dec_layers.extend(
            [
                HealpyChebyshev(
                    K=K,
                    Fout=fout,
                    activation=activation,
                    use_bn=use_bn,
                    use_bias=use_bias,
                ),
                HealpyChebyshev(
                    K=K,
                    Fout=fout,
                    activation=activation,
                    use_bn=use_bn,
                    use_bias=use_bias,
                ),
            ]
        )
        if dropout_rate > 0.0:
            dec_layers.append(Dropout(dropout_rate))

        self.body = HealpyGCNN(
            nside=nside,
            indices=indices,
            layers=dec_layers,
            n_neighbors=8,
            max_batch_size=max_batch_size,
            initial_Fin=fin,
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "nside": self.nside,
                "npix": self.npix,
                "fin": self.fin,
                "fout": self.fout,
                "activation": self.activation_fn,
                "K": self.K,
                "max_batch_size": self.max_batch_size,
                "upsample": self.upsample,
                "use_bn": self.use_bn,
                "use_bias": self.use_bias,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config

    def call(self, inputs, skip, training=False):
        x = self.body(inputs, training=training)
        if skip is not None:
            x = x + skip
        return x


@tf.keras.saving.register_keras_serializable()
class ResidualHealpyUNet:
    def __init__(
        self,
        input_shape,
        activation="sigmoid",
        max_batch_size=32,
        dropout_rate=0.1,
        use_bn=True,
        use_bias=True,
    ):
        self.input_shape = input_shape
        self.activation = activation
        self.max_batch_size = max_batch_size
        self.npix = input_shape[1]
        self.nside = hp.npix2nside(self.npix)
        self.npol = input_shape[2]
        self.dropout_rate = dropout_rate
        self.use_bn = use_bn
        self.use_bias = use_bias

    def get_config(self):
        return {
            "input_shape": self.input_shape,
            "activation": self.activation,
            "max_batch_size": self.max_batch_size,
            "dropout_rate": self.dropout_rate,
            "use_bn": self.use_bn,
            "use_bias": self.use_bias,
        }

    def get_model(self):
        depth = int(math.log2(self.nside))  # - 1

        base_channels = [self.npol] + [2 ** (i + 4) for i in range(depth)]
        level_npixels = [self.npix // (4**i) for i in range(depth + 1)]
        level_nsides = [hp.npix2nside(npix) for npix in level_npixels]
        Ks = [3 + (2 * (i // 2)) for i in range(depth + 1)]
        # Ks = [3 for _ in range(depth + 1)]
        logger.debug(f"Using Ks: {Ks}")

        inputs = tf.keras.Input(shape=self.input_shape[1:])
        x = inputs
        # x = layers.BatchNormalization()(x)

        skips = []
        for i in range(depth):
            block = EncoderBlock(
                level_nsides[i],
                level_npixels[i],
                fin=base_channels[i],
                fout=base_channels[i + 1],
                activation=self.activation,
                max_batch_size=self.max_batch_size,
                K=Ks[i],
                dropout_rate=self.dropout_rate,
                use_bias=self.use_bias,
                use_bn=self.use_bn,
            )
            x, skip = block(x)
            skips.append(skip)

        bottleneck = HealpyGCNN(
            nside=level_nsides[depth],
            indices=np.arange(level_npixels[depth]),
            layers=[
                HealpyChebyshev(
                    K=Ks[-1],
                    Fout=base_channels[-1],
                    activation=self.activation,
                    use_bias=self.use_bias,
                    use_bn=self.use_bn,
                ),
                HealpyChebyshev(
                    K=Ks[-1],
                    Fout=base_channels[-1],
                    activation=self.activation,
                    use_bias=self.use_bias,
                    use_bn=self.use_bn,
                ),
            ],
            n_neighbors=8,
            max_batch_size=self.max_batch_size,
            initial_Fin=base_channels[-1],
            name="bottleneck",
        )
        x = bottleneck(x)

        for i in reversed(range(1, depth + 1)):
            block = DecoderBlock(
                level_nsides[i],
                level_npixels[i],
                fin=base_channels[i],
                fout=base_channels[i],
                activation=self.activation,
                max_batch_size=self.max_batch_size,
                K=Ks[i],
                dropout_rate=self.dropout_rate,
                use_bias=self.use_bias,
                use_bn=self.use_bn,
            )
            x = block(x, skips[i - 1])

        # final block to account for upscaling to phi
        block = DecoderBlock(
            level_nsides[0],
            level_npixels[0],
            fin=base_channels[0],
            fout=base_channels[0],
            activation=None,
            max_batch_size=self.max_batch_size,
            K=Ks[0],
            dropout_rate=0.0,
            use_bias=self.use_bias,
            use_bn=self.use_bn,
        )
        x = block(x, None)

        # output
        output_head = HealpyGCNN(
            nside=self.nside,
            indices=np.arange(self.npix),
            layers=[
                HealpyChebyshev(
                    K=1,
                    Fout=base_channels[0],
                    activation=None,
                    use_bn=False,
                    use_bias=True,
                ),
            ],
            n_neighbors=8,
            max_batch_size=self.max_batch_size,
            initial_Fin=self.npol,
            name="output",
        )
        outputs = output_head(x)
        return tf.keras.Model(inputs, outputs)

# Load Configuration

In [ ]:
core = Core(
    [
        "settings/n64.json",
        "--nsims",
        "10000",
        "--phi_scale",
        "1",
        "--shapes",
        "local",
        "--fnl_range",
        "-1000",
        "1000",
    ]
)

date_time = "1763423947"
run_name = f"trainer-{date_time}"
logger.info(f"Run name: {run_name}")

save_dir = f"{core.dirs['model']}/{core.name}"
run_info = f"{core.shapes_str}-{run_name}"
plot_prefix = f"{core.dirs['plot']}/{core.name}/train/{core.slurm.job}"
os.makedirs(save_dir, exist_ok=True)
os.makedirs(f"{core.dirs['plot']}/{core.name}", exist_ok=True)
os.makedirs(os.path.dirname(plot_prefix), exist_ok=True)

# setups where we will
unet_model_path = f"{save_dir}/unet-{run_info}.keras"
fnl_model_path = f"{save_dir}/fnl-{run_info}.keras"

# Load Pre-trained Models

In [ ]:
# Load U-Net model for phi reconstruction
if os.path.exists(unet_model_path):
    logger.info(f"Loading U-Net model from: {unet_model_path}")
    u_net = tf.keras.models.load_model(unet_model_path)
    u_net.summary()
    print("U-Net model loaded successfully")
else:
    raise FileNotFoundError(f"U-Net model not found at {unet_model_path}")

# Load FNL model for non-Gaussianity detection
if os.path.exists(fnl_model_path):
    logger.info(f"Loading FNL model from: {fnl_model_path}")
    fnl_model = tf.keras.models.load_model(fnl_model_path)
    fnl_model.summary()
    print("FNL model loaded successfully")
else:
    raise FileNotFoundError(f"FNL model not found at {fnl_model_path}")

# Prepare Data and Normalization

In [ ]:
# Set up data loading parameters
batch_size = 32
data_fraction = 0.1

# Create PhiMapDataset for U-Net testing (lensed maps -> phi reconstruction)
ds_unet = PhiMapDataset.fromCore(core, lensed=True)
train_unet, val_unet, test_unet = ds_unet.split(
    train_size=0.8 * data_fraction,
    val_size=0.1 * data_fraction,
    test_size=0.1 * data_fraction,
    to_tf=True,
    batch_size=batch_size,
    duplicates=[25, 10, 2],
    gen_batch_size=4,
    cache_file="model_inspector_unet",  # Enable file-based caching
)

# Create MapDataset for FNL testing (unlensed maps)
ds_fnl = MapDataset.fromCore(core, lensed=False)
train_fnl, val_fnl, test_fnl = ds_fnl.split(
    train_size=0.4 * data_fraction,
    val_size=0.1 * data_fraction,
    test_size=0.5 * data_fraction,
    to_tf=True,
    batch_size=batch_size,
    duplicates=[25, 10, 2],
    gen_batch_size=4,
    cache_file="model_inspector_fnl",  # Enable file-based caching
)

ds_fnl = MapDataset.fromCore(core, lensed=True)
train_lens_fnl, val_lens_fnl, test_lens_fnl = ds_fnl.split(
    train_size=0.4 * data_fraction,
    val_size=0.1 * data_fraction,
    test_size=0.5 * data_fraction,
    to_tf=True,
    batch_size=batch_size,
    duplicates=[25, 10, 2],
    gen_batch_size=4,
    cache_file="model_inspector_fnl_lens",  # Enable file-based caching
)

print(f"U-Net test dataset created (file-cached)")
print(f"FNL test dataset created (file-cached)")

In [ ]:
# Create normalization layer for U-Net outputs
# This is important to undo the normalization that was applied during training
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(train_unet.map(lambda x, y: y))


def normalize_y(x, y):
    """Apply normalization to targets"""
    return x, normalizer(y)


def undo_norm(y):
    """Undo normalization to get original scale values"""
    if isinstance(y, np.ndarray):
        val = y * np.sqrt(normalizer.variance.numpy()) + normalizer.mean.numpy()
    else:
        val = y * tf.sqrt(normalizer.variance) + normalizer.mean
    return val


# Apply normalization to training/validation data
# train_unet_norm = train_unet.map(normalize_y)
# val_unet_norm = val_unet.map(normalize_y)

print(f"Normalization parameters:")
print(f"  Mean: {normalizer.mean.numpy()}")
print(f"  Variance: {normalizer.variance.numpy()}")
print(f"  Std Dev: {np.sqrt(normalizer.variance.numpy())}")

# U-Net Power Spectrum Analysis

In [ ]:
# Get test sample from U-Net dataset

# Extract test data ONCE - concatenate into single arrays
test_x, test_y = [], []
for x, y in test_unet:
    test_x.append(x.numpy())
    test_y.append(y.numpy())

test_phi_true = np.concatenate(test_y)
test_phi_true = test_phi_true[0, :, 0]

pred_phi = u_net.predict(np.concatenate(test_x), verbose=0)
pred_phi = undo_norm(pred_phi)
pred_phi = pred_phi[0, :, 0]

# Convert ordering for display
test_phi_ring = hp.reorder(test_phi_true, n2r=True)
pred_phi_ring = hp.reorder(pred_phi, n2r=True)

# Display true and predicted phi maps
for map_data, name in [(test_phi_ring, "True"), (pred_phi_ring, "Predicted")]:
    hp.mollview(
        map_data,
        title=f"{name} phi map",
        unit="phi",
        cmap="viridis",
        norm="hist",
    )
    plt.show()

# Display residual maps
hp.mollview(test_phi_true - pred_phi, title="Residual (Nest)", cmap="RdBu_r")
plt.show()

hp.mollview(test_phi_ring - pred_phi_ring, title="Residual (Ring)", cmap="RdBu_r")
plt.show()

In [ ]:
# Compute power spectra for both Ring and Nest orderings
plt.figure(figsize=(10, 5))
for map_data, name in [
    (test_phi_ring, "True (ring)"),
    (pred_phi_ring, "Predicted (ring)"),
]:
    cl = hp.anafast(
        map_data,
        # lmax=core.lmax,
        pol=False,
        use_pixel_weights=True,
    )
    plt.loglog(cl, label=name, linewidth=2)

plt.legend(fontsize=12)
plt.xlabel(r"$\ell$", fontsize=12)
plt.ylabel(r"$C_\ell^{\phi\phi}$", fontsize=12)
plt.title("U-Net Phi Power Spectrum Comparison (Ring Ordering)", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Nest ordering comparison
plt.figure(figsize=(10, 5))
for map_data, name in [(test_phi_true, "True (nest)"), (pred_phi, "Predicted (nest)")]:
    cl = hp.anafast(
        map_data,
        # lmax=core.lmax,
        pol=False,
        use_pixel_weights=True,
    )
    plt.loglog(cl, label=name, linewidth=2)

plt.legend(fontsize=12)
plt.xlabel(r"$\ell$", fontsize=12)
plt.ylabel(r"$C_\ell^{\phi\phi}$", fontsize=12)
plt.title("U-Net Phi Power Spectrum Comparison (Nest Ordering)", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Power spectrum of the lensed input map
# Extract first lensed input map from U-Net test dataset
first_unet_input = np.concatenate([x for x, y in test_unet])[0, :, 0]
first_unet_input_ring = hp.reorder(first_unet_input, n2r=True)

# Compute power spectrum
cl_unet_input = hp.anafast(
    first_unet_input_ring,
    # lmax=core.lmax,
    pol=False,
    use_pixel_weights=True,
)
ell_unet = np.arange(len(cl_unet_input))

plt.figure(figsize=(10, 6))
plt.loglog(
    ell_unet[2:],
    cl_unet_input[2:],
    linewidth=2.5,
    color="coral",
    label="Lensed Input Map",
)
plt.xlabel(r"$\ell$", fontsize=12)
plt.ylabel(r"$C_\ell$", fontsize=12)
plt.title("Power Spectrum of First U-Net Input (Lensed) Map", fontsize=13)
plt.grid(True, alpha=0.3, which="both")
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# FNL Model Predictions Analysis

In [ ]:
# Extract test data ONCE - concatenate into single arrays
test_x, test_y = [], []
for x, y in test_fnl:
    test_x.append(x.numpy())
    test_y.append(y.numpy())
fnl_test_data = np.concatenate(test_x)
fnl_truth = np.concatenate(test_y)

# Get test ground truth and make FNL predictions on unlensed maps
fnl_preds = fnl_model.predict(fnl_test_data, verbose=0)

# Flatten for analysis
fnl_truth_flat = fnl_truth.ravel()
fnl_preds_flat = fnl_preds.ravel()

# Calculate error metrics
fnl_error = fnl_preds_flat - fnl_truth_flat
mae = np.mean(np.abs(fnl_error))
rmse = np.sqrt(np.mean(fnl_error**2))

print(f"FNL Model Performance Metrics:")
print(f"  Mean Absolute Error: {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")

# Get expected uncertainty from core
sigma = core.get_likelihoods(False)[0]
print(f"  Expected 1-sigma: {sigma:.4f}")

In [ ]:
# Plot FNL prediction results
line = np.array([np.nanmin(fnl_truth_flat), np.nanmax(fnl_truth_flat)])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Scatter plot: predicted vs true
axes[0].scatter(fnl_truth_flat, fnl_preds_flat, alpha=0.5, s=20)
axes[0].plot(line, line, "r--", linewidth=2, label="Perfect prediction")
axes[0].plot(line, line + sigma, "g--", linewidth=2, label=f"+σ={sigma:.2f}")
axes[0].plot(line, line - sigma, "g--", linewidth=2, label=f"-σ={-sigma:.2f}")
axes[0].set_xlabel("True fNL", fontsize=12)
axes[0].set_ylabel("Predicted fNL", fontsize=12)
axes[0].set_title("fNL Prediction from Unlensed Maps", fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histogram: error distribution
axes[1].hist(fnl_error, bins=50, color="C0", alpha=0.7, edgecolor="black")
axes[1].axvline(sigma, color="g", linestyle="--", linewidth=2, label=f"+σ={sigma:.2f}")
axes[1].axvline(
    -sigma, color="g", linestyle="--", linewidth=2, label=f"-σ={-sigma:.2f}"
)
axes[1].axvline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[1].set_xlabel("Prediction Error", fontsize=12)
axes[1].set_ylabel("Count", fontsize=12)
axes[1].set_title("fNL Error Distribution", fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

# Line plot: predictions vs truth over test set
axes[2].plot(fnl_preds_flat, alpha=0.7, label="Predicted", linewidth=1.5)
axes[2].plot(fnl_truth_flat, alpha=0.7, label="True", linewidth=1.5)
axes[2].set_xlabel("Sample Index", fontsize=12)
axes[2].set_ylabel("fNL Value", fontsize=12)
axes[2].set_title("fNL Values Over Test Set", fontsize=12)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot distribution of test set ground truth values
vals = fnl_truth_flat

plt.figure(figsize=(8, 5))
plt.hist(vals, bins=50, color="C0", alpha=0.8, edgecolor="black")
plt.xlabel("fNL (ground truth)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title(f"Distribution of Test Set fNL Values (N={vals.size})", fontsize=14)
plt.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# Plot power spectrum of first test FNL input map
first_test_map = np.concatenate([x for x, y in test_fnl])[0, :, 0]
first_test_map_ring = hp.reorder(first_test_map, n2r=True)

# Compute power spectrum
cl_test = hp.anafast(
    first_test_map_ring,
    # lmax=core.lmax,
    pol=False,
    use_pixel_weights=True,
)
ell = np.arange(len(cl_test))

plt.figure(figsize=(10, 6))
plt.loglog(
    ell[2:], cl_test[2:], linewidth=2.5, color="steelblue", label="First Test Map"
)
plt.xlabel(r"$\ell$", fontsize=12)
plt.ylabel(r"$C_\ell$", fontsize=12)
plt.title("Power Spectrum of First FNL Test Input Map", fontsize=13)
plt.grid(True, alpha=0.3, which="both")
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Final Network: Delensing Pipeline (U-Net → FNL)

In [ ]:
# Create dataset with lensed maps (full sky, joint training scenario)
ds_final = MapDataset.fromCore(core, lensed=True)
train_final, val_final, test_final = ds_final.split(
    train_size=0.8 * data_fraction,
    val_size=0.1 * data_fraction,
    test_size=0.1 * data_fraction,
    to_tf=True,
    batch_size=batch_size,
    duplicates=[25, 10, 2],
    gen_batch_size=4,
    cache_file="model_inspector_final",
)

print(f"Final network dataset created (lensed maps)")

# Extract test data
lensed_test_maps = np.concatenate([x for x, y in test_final])
final_truth_fnl = np.concatenate([y for x, y in test_final])

print(f"Lensed test maps shape: {lensed_test_maps.shape}")
print(f"Final ground truth fNL shape: {final_truth_fnl.shape}")

In [ ]:
# Step 1: Run U-Net to estimate phi and delens the lensed maps
print("Running U-Net on lensed maps to estimate phi...")
phi_pred_final = u_net.predict(lensed_test_maps, verbose=0)
phi_pred_final = undo_norm(phi_pred_final)  # * 1e2

# Step 2: Delens the lensed maps using the predicted phi
# First need to set up the delensing pipeline similar to trainer.ipynb
res_arcmin = hp.nside2resol(core.nside, arcmin=True)
res_rad = res_arcmin * utils.arcmin

# Snap to nearest resolution that evenly divides the sky
ny = int(round(np.pi / res_rad))
res_fixed = np.pi / ny
print(
    f"HEALPy resolution (arcmin): {res_arcmin:.6f}, Fixed resolution (arcmin): {res_fixed / utils.arcmin:.6f}"
)

# Build geometry with fixed resolution
shape, wcs = enmap.fullsky_geometry(res_fixed)


def _delens_batch(lensed, phi, nside=core.nside):
    """Delens a batch of lensed maps given their phi estimates"""
    delensed_maps = np.zeros_like(lensed)
    for i, (l_map, p_map) in enumerate(zip(lensed, phi)):
        # Convert from NEST to RING ordering for pixell operations
        l_map_ring = hp.reorder(l_map[:, 0], n2r=True).astype(np.float32)
        p_map_ring = hp.reorder(p_map[:, 0], n2r=True).astype(np.float32)

        # Reproject to flat-sky geometry
        lensed_enmap = reproject.healpix2map(l_map_ring, shape, wcs)
        phi_enmap = reproject.healpix2map(p_map_ring, shape, wcs)

        # Delens the map
        delensed_enmap = lensing.delens_map(lensed_enmap, phi_enmap)

        # Reproject back to HEALPix
        m = reproject.map2healpix(delensed_enmap, nside)
        # m = hp.remove_dipole(m, copy=False)
        m = hp.reorder(m, r2n=True).astype(np.float32)
        delensed_maps[i, :, 0] = m

    return delensed_maps


# Delens the maps in batches
delensed_final = _delens_batch(lensed_test_maps, phi_pred_final)

In [ ]:
# Step 3: Create a dataset from the delensed maps
delensed_ds = tf.data.Dataset.from_tensor_slices(delensed_final).batch(batch_size)

print("Running FNL model on delensed maps...")
fnl_pred_final = fnl_model.predict(delensed_ds, verbose=0)

print(f"Final FNL predictions shape: {fnl_pred_final.shape}")

# Flatten for analysis
fnl_pred_final_flat = fnl_pred_final.ravel()
final_truth_fnl_flat = final_truth_fnl.ravel()

# Calculate error metrics for the full pipeline
fnl_error_final = fnl_pred_final_flat - final_truth_fnl_flat
mae_final = np.mean(np.abs(fnl_error_final))
rmse_final = np.sqrt(np.mean(fnl_error_final**2))

print(f"\nFinal Network (Lensed → U-Net → FNL) Performance:")
print(f"  Mean Absolute Error: {mae_final:.4f}")
print(f"  RMSE: {rmse_final:.4f}")
print(f"  Expected 1-sigma: {sigma:.4f}")

# Compare with direct FNL on unlensed maps
print(f"\nComparison:")
print(f"  Direct FNL on unlensed maps MAE: {mae:.4f}")
print(f"  Final network MAE: {mae_final:.4f}")

In [ ]:
# Plot 1: Final network predictions vs truth (3-panel figure)
line_final = np.array(
    [np.nanmin(final_truth_fnl_flat), np.nanmax(final_truth_fnl_flat)]
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Scatter plot: predicted vs true
axes[0].scatter(final_truth_fnl_flat, fnl_pred_final_flat, alpha=0.5, s=20)
axes[0].plot(line_final, line_final, "r--", linewidth=2, label="Perfect prediction")
axes[0].plot(
    line_final, line_final + sigma, "g--", linewidth=2, label=f"+σ={sigma:.2f}"
)
axes[0].plot(
    line_final, line_final - sigma, "g--", linewidth=2, label=f"-σ={-sigma:.2f}"
)
axes[0].set_xlabel("True fNL (lensed)", fontsize=12)
axes[0].set_ylabel("Predicted fNL (from delensed)", fontsize=12)
axes[0].set_title("Final Network: Lensed → U-Net → Delensed → FNL", fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histogram: error distribution
axes[1].hist(fnl_error_final, bins=50, color="C1", alpha=0.7, edgecolor="black")
axes[1].axvline(sigma, color="g", linestyle="--", linewidth=2, label=f"+σ={sigma:.2f}")
axes[1].axvline(
    -sigma, color="g", linestyle="--", linewidth=2, label=f"-σ={-sigma:.2f}"
)
axes[1].axvline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[1].set_xlabel("Prediction Error", fontsize=12)
axes[1].set_ylabel("Count", fontsize=12)
axes[1].set_title("Error Distribution (Full Pipeline)", fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

# Line plot: predictions vs truth over test set
axes[2].plot(
    fnl_pred_final_flat, alpha=0.7, label="Predicted (from delensed)", linewidth=1.5
)
axes[2].plot(final_truth_fnl_flat, alpha=0.7, label="True", linewidth=1.5)
axes[2].set_xlabel("Sample Index", fontsize=12)
axes[2].set_ylabel("fNL Value", fontsize=12)
axes[2].set_title("fNL Values Over Full Test Set", fontsize=12)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

new

In [ ]:
fnl_lens_out = fnl_model.predict(test_lens_fnl, verbose=1)

In [ ]:
fnl_lens_truth = np.concatenate([y for x, y in test_lens_fnl])
plt.plot(fnl_lens_truth, fnl_lens_out, "o", alpha=0.3)
plt.show()

In [ ]:
# Plot 2: Comparison of all three models
# Direct FNL (unlensed), Direct FNL (lensed via U-Net), and Final network
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Row 1: Unlensed data FNL model
axes[0, 0].scatter(fnl_truth_flat, fnl_preds_flat, alpha=0.5, s=20, c="C0")
axes[0, 0].plot(line, line, "r--", linewidth=2)
axes[0, 0].set_xlabel("True fNL", fontsize=11)
axes[0, 0].set_ylabel("Predicted fNL", fontsize=11)
axes[0, 0].set_title(f"FNL Direct (Unlensed)\nMAE={mae:.4f}", fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(fnl_error, bins=40, color="C0", alpha=0.7, edgecolor="black")
axes[0, 1].axvline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[0, 1].set_xlabel("Error", fontsize=11)
axes[0, 1].set_ylabel("Count", fontsize=11)
axes[0, 1].set_title("Error Distribution", fontsize=11)
axes[0, 1].grid(True, alpha=0.3, axis="y")

axes[0, 2].scatter(
    np.arange(len(fnl_preds_flat[:100])), fnl_error[:100], alpha=0.6, s=20, c="C0"
)
axes[0, 2].axhline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[0, 2].set_xlabel("Sample Index", fontsize=11)
axes[0, 2].set_ylabel("Prediction Error", fontsize=11)
axes[0, 2].set_title("Error vs Sample (first 100)", fontsize=11)
axes[0, 2].grid(True, alpha=0.3)

# Row 2: Final network (lensed → U-Net → delensed → FNL)
axes[1, 0].scatter(final_truth_fnl_flat, fnl_pred_final_flat, alpha=0.5, s=20, c="C1")
axes[1, 0].plot(line_final, line_final, "r--", linewidth=2)
axes[1, 0].set_xlabel("True fNL (lensed)", fontsize=11)
axes[1, 0].set_ylabel("Predicted fNL", fontsize=11)
axes[1, 0].set_title(
    f"Final Network (Lensed→U-Net→FNL)\nMAE={mae_final:.4f}", fontsize=11
)
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(fnl_error_final, bins=40, color="C1", alpha=0.7, edgecolor="black")
axes[1, 1].axvline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[1, 1].set_xlabel("Error", fontsize=11)
axes[1, 1].set_ylabel("Count", fontsize=11)
axes[1, 1].set_title("Error Distribution", fontsize=11)
axes[1, 1].grid(True, alpha=0.3, axis="y")

axes[1, 2].scatter(
    np.arange(len(fnl_pred_final_flat[:100])),
    fnl_error_final[:100],
    alpha=0.6,
    s=20,
    c="C1",
)
axes[1, 2].axhline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[1, 2].set_xlabel("Sample Index", fontsize=11)
axes[1, 2].set_ylabel("Prediction Error", fontsize=11)
axes[1, 2].set_title("Error vs Sample (first 100)", fontsize=11)
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle(
    "Comparison: Direct FNL (Unlensed) vs Final Network (Lensed)", fontsize=14, y=1.00
)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Visualize example delensed map from the pipeline
# Extract first sample
first_lensed = lensed_test_maps[0, :, 0]
first_delensed = delensed_final[0, :, 0]
first_phi = phi_pred_final[0, :, 0]

# Convert to RING ordering for display
first_lensed_ring = hp.reorder(first_lensed, n2r=True)
first_delensed_ring = hp.reorder(first_delensed, n2r=True)
first_phi_ring = hp.reorder(first_phi, n2r=True)

fig = plt.figure(figsize=(14, 4))

# Lensed map
ax1 = fig.add_subplot(131, projection="mollweide")
plt.axes(ax1)
hp.mollview(first_lensed_ring, title="Input: Lensed Map", cmap="viridis", hold=True)

# Estimated phi
ax2 = fig.add_subplot(132, projection="mollweide")
plt.axes(ax2)
hp.mollview(
    first_phi_ring, title="U-Net Output: Estimated φ", cmap="viridis", hold=True
)

# Delensed map
ax3 = fig.add_subplot(133, projection="mollweide")
plt.axes(ax3)
hp.mollview(
    first_delensed_ring,
    title="Pipeline Output: Delensed Map",
    cmap="viridis",
    hold=True,
)

plt.tight_layout()
plt.show()

# Power spectra comparison
print("\nPower spectra analysis for first sample:")
cl_lensed = hp.anafast(first_lensed, lmax=core.lmax, pol=False)
cl_delensed = hp.anafast(first_delensed, lmax=core.lmax, pol=False)
ell = np.arange(len(cl_lensed))

plt.figure(figsize=(10, 6))
plt.loglog(ell[2:], cl_lensed[2:], label="Lensed", linewidth=2)
plt.loglog(ell[2:], cl_delensed[2:], label="Delensed by Pipeline", linewidth=2)
plt.xlabel(r"$\ell$", fontsize=12)
plt.ylabel(r"$C_\ell$", fontsize=12)
plt.title("Power Spectra: Lensed vs Delensed (First Sample)", fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("\n" + "=" * 70)
print("FINAL NETWORK SUMMARY: Lensed Maps → U-Net Delensing → fNL Prediction")
print("=" * 70)

print("\nStep 1 - U-Net Phi Estimation:")
print(f"  Input shape: {lensed_test_maps.shape} (lensed CMB maps)")
print(f"  Output shape: {phi_pred_final.shape} (estimated lensing potential)")
print(f"  Normalization: Applied during training, undone with learned statistics")

print("\nStep 2 - Map Delensing:")
print(f"  Method: Delensing with pixell using flat-sky approximation")
print(
    f"  HEALPix resolution: {res_arcmin:.6f} arcmin → {res_fixed / utils.arcmin:.6f} arcmin (fixed)"
)
print(f"  Output: {delensed_final.shape} (delensed CMB maps)")

print("\nStep 3 - FNL Prediction:")
print(
    f"  Input shape: {delensed_ds.cardinality().numpy() * batch_size} samples of delensed maps"
)
print(f"  Model: SCN-based fNL estimator trained on unlensed maps")
print(f"  Shapes detected: {core.shapes}")

print("\n" + "-" * 70)
print("PERFORMANCE COMPARISON:")
print("-" * 70)
print(f"{'Scenario':<40} {'MAE':>10} {'RMSE':>10}")
print("-" * 70)
print(
    f"{'Direct FNL (unlensed)':<40} {mae:>10.4f} {np.sqrt(np.mean(fnl_error**2)):>10.4f}"
)
print(
    f"{'Final Network (lensed→delensed→FNL)':<40} {mae_final:>10.4f} {np.sqrt(np.mean(fnl_error_final**2)):>10.4f}"
)
print(f"{'Expected 1-σ uncertainty':<40} {sigma:>10.4f}")
print("-" * 70)

improvement = ((mae - mae_final) / mae) * 100 if mae > 0 else 0
print(f"\nMAE improvement with delensing: {improvement:+.2f}%")
print(f"Number of test samples: {len(final_truth_fnl_flat)}")

# fine tuned network

In [ ]:
stop

In [ ]:
strategy = tf.distribute.MirroredStrategy()

print("\nLoading pre-trained models...")
print(f"Loading U-Net from: {unet_model_path}")
u_net_loaded = tf.keras.models.load_model(unet_model_path)

print(f"Loading FNL model from: {fnl_model_path}")
fnl_model_loaded = tf.keras.models.load_model(fnl_model_path)

In [ ]:
data_fraction = 0.01
batch_size = 32

ds_final = MapDataset.fromCore(core, lensed=True)
train_final, val_final, test_final = ds_final.split(
    train_size=0.8 * data_fraction,
    val_size=0.1 * data_fraction,
    test_size=0.1 * data_fraction,
    to_tf=True,
    batch_size=batch_size,
    duplicates=[25, 10, 2],
    gen_batch_size=4,
    cache_file="model_inspector_final",
)

# Extract training data from dataset
train_lensed_batch = []
train_fnl_labels = []
for x, y in train_final:
    train_lensed_batch.append(x.numpy())
    train_fnl_labels.append(y.numpy())
train_lensed_data = np.concatenate(train_lensed_batch)
train_fnl_truth = np.concatenate(train_fnl_labels)

In [ ]:
# Step 2: Delens the lensed maps using the predicted phi
# First need to set up the delensing pipeline similar to trainer.ipynb
res_arcmin = hp.nside2resol(core.nside, arcmin=True)
res_rad = res_arcmin * utils.arcmin

# Snap to nearest resolution that evenly divides the sky
ny = int(round(np.pi / res_rad))
res_fixed = np.pi / ny
print(
    f"HEALPy resolution (arcmin): {res_arcmin:.6f}, Fixed resolution (arcmin): {res_fixed / utils.arcmin:.6f}"
)

# Build geometry with fixed resolution
shape, wcs = enmap.fullsky_geometry(res_fixed)


def _delens_batch(lensed, phi, nside=core.nside):
    """Delens a batch of lensed maps given their phi estimates"""
    delensed_maps = np.zeros_like(lensed)
    for i, (l_map, p_map) in enumerate(zip(lensed, phi)):
        # Convert from NEST to RING ordering for pixell operations
        l_map_ring = hp.reorder(l_map[:, 0], n2r=True).astype(np.float32)
        p_map_ring = hp.reorder(p_map[:, 0], n2r=True).astype(np.float32)

        # Reproject to flat-sky geometry
        lensed_enmap = reproject.healpix2map(l_map_ring, shape, wcs)
        phi_enmap = reproject.healpix2map(p_map_ring, shape, wcs)

        # Delens the map
        delensed_enmap = lensing.delens_map(lensed_enmap, phi_enmap)

        # Reproject back to HEALPix
        m = reproject.map2healpix(delensed_enmap, nside)
        # m = hp.remove_dipole(m, copy=False)
        m = hp.reorder(m, r2n=True).astype(np.float32)
        delensed_maps[i, :, 0] = m

    return delensed_maps

In [ ]:
phi_pred_train = u_net_loaded.predict(train_lensed_data, verbose=1)
phi_pred_train = undo_norm(phi_pred_train)

delensed_train = _delens_batch(train_lensed_data, phi_pred_train)


# Create TensorFlow dataset from delensed maps
train_ds_finetune = (
    tf.data.Dataset.from_tensor_slices((delensed_train, train_fnl_truth))
    .batch(batch_size)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

# Extract and process validation data similarly
print("\nProcessing Validation Data...")
val_lensed_batch = []
val_fnl_labels = []
for x, y in val_final:
    val_lensed_batch.append(x.numpy())
    val_fnl_labels.append(y.numpy())
val_lensed_data = np.concatenate(val_lensed_batch)
val_fnl_truth = np.concatenate(val_fnl_labels)

# Run U-Net on validation data
phi_pred_val = u_net_loaded.predict(val_lensed_data, verbose=1)
phi_pred_val = undo_norm(phi_pred_val)

# Delens validation maps
delensed_val = _delens_batch(val_lensed_data, phi_pred_val)
print(f"Validation delensed maps shape: {delensed_val.shape}")

val_ds_finetune = (
    tf.data.Dataset.from_tensor_slices((delensed_val, val_fnl_truth))
    .batch(batch_size)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# All training code inside strategy scope
with strategy.scope():
    # Prepare optimizer and callbacks
    lr_schedule = ExponentialDecay(
        initial_learning_rate=1e-4, decay_steps=10, decay_rate=0.95
    )
    optimizer = AdamW(learning_rate=lr_schedule)

    fnl_model_loaded.compile(optimizer=optimizer, loss="mse", metrics=["mae"])

    # Define callbacks
    callbacks = [
        EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
        ),
        TerminateOnNaN(),
    ]

    # Train the model with distributed strategy
    print("\nStarting distributed fine-tuning of FNL model...")
    history = fnl_model_loaded.fit(
        train_ds_finetune,
        validation_data=val_ds_finetune,
        epochs=20,
        callbacks=callbacks,
        verbose=1,
    )

print("\nFine-tuning completed!")
print(f"Best validation loss: {np.min(history.history['val_loss']):.6f}")
print(f"Final training loss: {history.history['loss'][-1]:.6f}")

In [ ]:
# Evaluate on Test Set
print("\n" + "=" * 70)
print("Evaluating Fine-tuned FNL Model on Test Set")
print("=" * 70)

# Extract test data
test_lensed_batch = []
test_fnl_labels = []
for x, y in test_final:
    test_lensed_batch.append(x.numpy())
    test_fnl_labels.append(y.numpy())
test_lensed_data = np.concatenate(test_lensed_batch)
test_fnl_truth = np.concatenate(test_fnl_labels)

# Run U-Net on test data
print("\nRunning U-Net phi prediction on test data...")
phi_pred_test = u_net_loaded.predict(test_lensed_data, verbose=1)
phi_pred_test = undo_norm(phi_pred_test)

# Delens test maps
print("Delensing test maps...")
delensed_test = _delens_batch(test_lensed_data, phi_pred_test)

# Make predictions on delensed test data
test_ds_finetune = (
    tf.data.Dataset.from_tensor_slices(delensed_test)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

fnl_pred_finetuned = fnl_model_loaded.predict(test_ds_finetune, verbose=1)

# Calculate metrics
fnl_pred_finetuned_flat = fnl_pred_finetuned.ravel()
fnl_truth_flat = test_fnl_truth.ravel()
fnl_error_finetuned = fnl_pred_finetuned_flat - fnl_truth_flat
mae_finetuned = np.mean(np.abs(fnl_error_finetuned))
rmse_finetuned = np.sqrt(np.mean(fnl_error_finetuned**2))

print(f"\nFine-tuned FNL Model Performance on Test Set:")
print(f"  Mean Absolute Error: {mae_finetuned:.4f}")
print(f"  RMSE: {rmse_finetuned:.4f}")
print(f"  Expected 1-sigma: {sigma:.4f}")
print(f"  Prediction std: {np.std(fnl_pred_finetuned_flat):.4f}")
print(f"  Truth std: {np.std(fnl_truth_flat):.4f}")

In [ ]:
# Plot Training History
print("\nPlotting training history...")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss plot
axes[0].plot(history.history["loss"], label="Training Loss", linewidth=2)
axes[0].plot(history.history["val_loss"], label="Validation Loss", linewidth=2)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss (MSE)", fontsize=12)
axes[0].set_title("Fine-tuned FNL Model - Loss", fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale("log")

# MAE plot
axes[1].plot(history.history["mae"], label="Training MAE", linewidth=2)
axes[1].plot(history.history["val_mae"], label="Validation MAE", linewidth=2)
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("MAE", fontsize=12)
axes[1].set_title("Fine-tuned FNL Model - MAE", fontsize=12)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 1: Fine-tuned Model Predictions vs Truth (3-panel figure)
print("Plotting test set results...")
line_finetuned = np.array([np.nanmin(fnl_truth_flat), np.nanmax(fnl_truth_flat)])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Scatter plot: predicted vs true
axes[0].scatter(fnl_truth_flat, fnl_pred_finetuned_flat, alpha=0.5, s=20, color="C3")
axes[0].plot(
    line_finetuned, line_finetuned, "r--", linewidth=2, label="Perfect prediction"
)
axes[0].plot(
    line_finetuned,
    line_finetuned + sigma,
    "g--",
    linewidth=2,
    label=f"+σ={sigma:.2f}",
)
axes[0].plot(
    line_finetuned,
    line_finetuned - sigma,
    "g--",
    linewidth=2,
    label=f"-σ={-sigma:.2f}",
)
axes[0].set_xlabel("True fNL", fontsize=12)
axes[0].set_ylabel("Predicted fNL", fontsize=12)
axes[0].set_title(f"Fine-tuned FNL Model\nMAE={mae_finetuned:.4f}", fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Histogram: error distribution
axes[1].hist(fnl_error_finetuned, bins=50, color="C3", alpha=0.7, edgecolor="black")
axes[1].axvline(sigma, color="g", linestyle="--", linewidth=2, label=f"+σ={sigma:.2f}")
axes[1].axvline(
    -sigma, color="g", linestyle="--", linewidth=2, label=f"-σ={-sigma:.2f}"
)
axes[1].axvline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[1].set_xlabel("Prediction Error", fontsize=12)
axes[1].set_ylabel("Count", fontsize=12)
axes[1].set_title("Error Distribution", fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis="y")

# Line plot: predictions vs truth over test set
axes[2].plot(
    fnl_pred_finetuned_flat, alpha=0.7, label="Predicted (fine-tuned)", linewidth=1.5
)
axes[2].plot(fnl_truth_flat, alpha=0.7, label="True", linewidth=1.5)
axes[2].set_xlabel("Sample Index", fontsize=12)
axes[2].set_ylabel("fNL Value", fontsize=12)
axes[2].set_title("fNL Values Over Test Set", fontsize=12)
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTest set visualization complete!")

In [ ]:
# Performance Summary and Comparison
print("\n" + "=" * 80)
print("FINE-TUNED FNL MODEL: PERFORMANCE SUMMARY")
print("=" * 80)

print("\nTraining Configuration:")
print(f"  Optimizer: AdamW with exponential decay (lr_init=1e-4, decay_rate=0.95)")
print(f"  Loss function: MSE")
print(f"  Batch size: {batch_size}")
print(f"  Epochs trained: {len(history.history['loss'])}")
print(f"  Early stopping patience: 10 epochs")

print("\nData Processing Pipeline:")
print(f"  Stage 1: U-Net phi estimation on lensed training maps")
print(f"  Stage 2: Delensing using predicted phi with pixell")
print(f"  Stage 3: Fine-tuning FNL model on delensed maps")

print("\n" + "-" * 80)
print("TEST SET PERFORMANCE:")
print("-" * 80)

rmse_finetuned = np.sqrt(np.mean(fnl_error_finetuned**2))
print(f"  Mean Absolute Error: {mae_finetuned:.4f}")
print(f"  RMSE: {rmse_finetuned:.4f}")
print(f"  Expected 1-sigma: {sigma:.4f}")
print(f"  Prediction std: {np.std(fnl_pred_finetuned_flat):.4f}")
print(f"  Truth std: {np.std(fnl_truth_flat):.4f}")

print(f"\nError Statistics:")
print(f"  Mean error (bias): {np.mean(fnl_error_finetuned):.4f}")
print(
    f"  Error range: [{np.min(fnl_error_finetuned):.4f}, {np.max(fnl_error_finetuned):.4f}]"
)
print(
    f"  % within 1σ: {100 * np.sum(np.abs(fnl_error_finetuned) <= sigma) / len(fnl_error_finetuned):.1f}%"
)
print(
    f"  % within 2σ: {100 * np.sum(np.abs(fnl_error_finetuned) <= 2*sigma) / len(fnl_error_finetuned):.1f}%"
)

print(f"\nTest Set Statistics:")
print(f"  Number of samples: {len(fnl_truth_flat)}")
print(f"  True fNL range: [{np.min(fnl_truth_flat):.4f}, {np.max(fnl_truth_flat):.4f}]")
print(
    f"  Predicted range: [{np.min(fnl_pred_finetuned_flat):.4f}, {np.max(fnl_pred_finetuned_flat):.4f}]"
)

print("\n" + "=" * 80)

In [ ]:
# Comparison: Baseline (Unlensed) vs Fine-tuned (Lensed → Delensed)
print("\n" + "=" * 80)
print("COMPARISON: BASELINE vs FINE-TUNED MODEL")
print("=" * 80)

print("\nBaseline Model (trained on unlensed maps, no fine-tuning):")
print(f"  MAE: {mae:.4f}")
print(f"  RMSE: {np.sqrt(np.mean(fnl_error**2)):.4f}")

print("\nFine-tuned Model (U-Net → Delens → FNL on lensed training data):")
print(f"  MAE: {mae_finetuned:.4f}")
print(f"  RMSE: {rmse_finetuned:.4f}")

improvement = ((mae - mae_finetuned) / mae) * 100 if mae > 0 else 0
print(f"\nImprovement with fine-tuning: {improvement:+.2f}%")

print("\n" + "=" * 80)

In [ ]:
# Plot 2: Comprehensive Comparison - Baseline vs Fine-tuned
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Row 1: Baseline (unlensed) FNL model
line_baseline = np.array([np.nanmin(fnl_truth_flat), np.nanmax(fnl_truth_flat)])

axes[0, 0].scatter(fnl_truth_flat, fnl_preds_flat, alpha=0.5, s=20, c="C0")
axes[0, 0].plot(line_baseline, line_baseline, "r--", linewidth=2)
axes[0, 0].set_xlabel("True fNL", fontsize=11)
axes[0, 0].set_ylabel("Predicted fNL", fontsize=11)
axes[0, 0].set_title(f"Baseline (Unlensed)\nMAE={mae:.4f}", fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(fnl_error, bins=40, color="C0", alpha=0.7, edgecolor="black")
axes[0, 1].axvline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[0, 1].set_xlabel("Error", fontsize=11)
axes[0, 1].set_ylabel("Count", fontsize=11)
axes[0, 1].set_title("Error Distribution", fontsize=11)
axes[0, 1].grid(True, alpha=0.3, axis="y")

axes[0, 2].scatter(
    np.arange(len(fnl_preds_flat[:100])), fnl_error[:100], alpha=0.6, s=20, c="C0"
)
axes[0, 2].axhline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[0, 2].axhline(
    sigma, color="g", linestyle="--", linewidth=1.5, alpha=0.7, label=f"±σ={sigma:.2f}"
)
axes[0, 2].axhline(-sigma, color="g", linestyle="--", linewidth=1.5, alpha=0.7)
axes[0, 2].set_xlabel("Sample Index", fontsize=11)
axes[0, 2].set_ylabel("Prediction Error", fontsize=11)
axes[0, 2].set_title("Error vs Sample (first 100)", fontsize=11)
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].legend(fontsize=9)

# Row 2: Fine-tuned Model (lensed → delensed)
axes[1, 0].scatter(fnl_truth_flat, fnl_pred_finetuned_flat, alpha=0.5, s=20, c="C3")
axes[1, 0].plot(line_finetuned, line_finetuned, "r--", linewidth=2)
axes[1, 0].set_xlabel("True fNL", fontsize=11)
axes[1, 0].set_ylabel("Predicted fNL", fontsize=11)
axes[1, 0].set_title(
    f"Fine-tuned (Lensed→Delensed)\nMAE={mae_finetuned:.4f}", fontsize=11
)
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(fnl_error_finetuned, bins=40, color="C3", alpha=0.7, edgecolor="black")
axes[1, 1].axvline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[1, 1].set_xlabel("Error", fontsize=11)
axes[1, 1].set_ylabel("Count", fontsize=11)
axes[1, 1].set_title("Error Distribution", fontsize=11)
axes[1, 1].grid(True, alpha=0.3, axis="y")

axes[1, 2].scatter(
    np.arange(len(fnl_pred_finetuned_flat[:100])),
    fnl_error_finetuned[:100],
    alpha=0.6,
    s=20,
    c="C3",
)
axes[1, 2].axhline(0, color="r", linestyle="-", linewidth=1, alpha=0.5)
axes[1, 2].axhline(
    sigma, color="g", linestyle="--", linewidth=1.5, alpha=0.7, label=f"±σ={sigma:.2f}"
)
axes[1, 2].axhline(-sigma, color="g", linestyle="--", linewidth=1.5, alpha=0.7)
axes[1, 2].set_xlabel("Sample Index", fontsize=11)
axes[1, 2].set_ylabel("Prediction Error", fontsize=11)
axes[1, 2].set_title("Error vs Sample (first 100)", fontsize=11)
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].legend(fontsize=9)

plt.suptitle(
    "Comparison: Baseline (Unlensed) vs Fine-tuned Model (Lensed→Delensed)",
    fontsize=14,
    y=1.00,
)
plt.tight_layout()
plt.show()

print("Comparison plot complete!")

In [ ]:
# Plot 3: Example Predictions from Fine-tuned Model
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Select 4 diverse examples from the test set
n_samples = len(fnl_truth_flat)
indices = [0, n_samples // 3, 2 * n_samples // 3, n_samples - 1]

for idx, (ax, sample_idx) in enumerate(zip(axes.flat, indices)):
    true_val = fnl_truth_flat[sample_idx]
    pred_val = fnl_pred_finetuned_flat[sample_idx]
    error_val = fnl_error_finetuned[sample_idx]

    # Create bar plot comparing true vs predicted
    x_pos = [0, 1]
    values = [true_val, pred_val]
    colors = ["C0", "C3"]

    bars = ax.bar(
        x_pos, values, color=colors, alpha=0.7, edgecolor="black", linewidth=2
    )

    # Add error bands
    ax.axhspan(
        true_val - sigma, true_val + sigma, alpha=0.1, color="C0", label=f"±σ (1σ)"
    )

    # Customize
    ax.set_xticks(x_pos)
    ax.set_xticklabels(["True fNL", "Predicted fNL"], fontsize=11)
    ax.set_ylabel("fNL Value", fontsize=11)
    ax.set_title(
        f"Sample {sample_idx}\nTrue={true_val:.3f}, Pred={pred_val:.3f}, Error={error_val:+.3f}",
        fontsize=11,
    )
    ax.grid(True, alpha=0.3, axis="y")

    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold",
        )

    if idx == 0:
        ax.legend(loc="upper left", fontsize=9)

plt.suptitle(
    "Example Predictions from Fine-tuned FNL Model", fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.show()

print("Example predictions visualization complete!")

In [ ]:
# Final Summary Report
print("\n" + "=" * 80)
print("FINAL ANALYSIS: FINE-TUNING LENSED MAP FNL PREDICTION")
print("=" * 80)

print("\n1. WORKFLOW SUMMARY")
print("-" * 80)
print("Step 1: Load Pre-trained U-Net (phi predictor) and FNL Model")
print("Step 2: Process Training Data:")
print("        - Extract lensed CMB maps from training set")
print("        - Run U-Net to predict lensing potential (phi)")
print("        - Delens maps using predicted phi (pixell)")
print("Step 3: Fine-tune FNL Model on delensed training maps")
print("Step 4: Evaluate on test set (lensed → delensed → predict fNL)")
print("Step 5: Compare baseline vs fine-tuned performance")

print("\n2. DATA PIPELINE")
print("-" * 80)
print(f"Training samples processed: {len(train_lensed_data)}")
print(f"Validation samples processed: {len(val_lensed_data)}")
print(f"Test samples processed: {len(test_lensed_data)}")
print(f"Resolution: n={core.nside} ({core.npix} pixels)")
print(f"Shapes: {core.shapes}")

print("\n3. MODEL ARCHITECTURE")
print("-" * 80)
print("U-Net Model:")
print(f"  - Input: Lensed maps (shape: {u_net_loaded.input_shape})")
print(f"  - Output: Phi predictions (shape: {u_net_loaded.output_shape})")
print(f"  - Status: Frozen (not retrained)")
print("\nFNL Predictor Model:")
print(f"  - Input: Unlensed/delensed maps (shape: {fnl_model_loaded.input_shape})")
print(f"  - Output: fNL scalar value")
print(f"  - Status: Fine-tuned on delensed training data")

print("\n4. TRAINING CONFIGURATION")
print("-" * 80)
print(f"Optimizer: AdamW (learning rate schedule: exponential decay)")
print(f"Initial learning rate: 1e-4")
print(f"Decay rate: 0.95 per 10 steps")
print(f"Loss function: Mean Squared Error (MSE)")
print(f"Batch size: {batch_size}")
print(f"Epochs trained: {len(history.history['loss'])}")
print(f"Early stopping: Yes (patience=10)")
print(f"Best validation loss: {np.min(history.history['val_loss']):.6f}")

print("\n5. PERFORMANCE COMPARISON")
print("-" * 80)
print(f"{'Metric':<30} {'Baseline':>15} {'Fine-tuned':>15}")
print("-" * 80)
print(f"{'MAE (Mean Absolute Error)':<30} {mae:>15.4f} {mae_finetuned:>15.4f}")
print(
    f"{'RMSE (Root Mean Sq Error)':<30} {np.sqrt(np.mean(fnl_error**2)):>15.4f} {rmse_finetuned:>15.4f}"
)
print(f"{'Expected 1-sigma':<30} {sigma:>15.4f} {sigma:>15.4f}")
print(
    f"{'% within 1σ':<30} {100*np.sum(np.abs(fnl_error) <= sigma) / len(fnl_error):>14.1f}% {100*np.sum(np.abs(fnl_error_finetuned) <= sigma) / len(fnl_error_finetuned):>14.1f}%"
)
print(
    f"{'% within 2σ':<30} {100*np.sum(np.abs(fnl_error) <= 2*sigma) / len(fnl_error):>14.1f}% {100*np.sum(np.abs(fnl_error_finetuned) <= 2*sigma) / len(fnl_error_finetuned):>14.1f}%"
)

improvement = ((mae - mae_finetuned) / mae) * 100 if mae > 0 else 0
print("\n" + "-" * 80)
if improvement > 0:
    print(f"✓ IMPROVEMENT: Fine-tuning improved MAE by {improvement:.2f}%")
else:
    print(
        f"✗ NO IMPROVEMENT: Fine-tuning degraded performance by {abs(improvement):.2f}%"
    )

print("\n6. TEST SET STATISTICS")
print("-" * 80)
print(f"Number of test samples: {len(fnl_truth_flat)}")
print(f"True fNL range: [{np.min(fnl_truth_flat):.2f}, {np.max(fnl_truth_flat):.2f}]")
print(
    f"Predicted range: [{np.min(fnl_pred_finetuned_flat):.2f}, {np.max(fnl_pred_finetuned_flat):.2f}]"
)
print(f"Prediction bias (mean error): {np.mean(fnl_error_finetuned):.4f}")
print(f"Prediction std: {np.std(fnl_pred_finetuned_flat):.4f}")
print(f"Truth std: {np.std(fnl_truth_flat):.4f}")

print("\n" + "=" * 80)
print("Analysis complete!")
print("=" * 80)

In [ ]:
# Power Spectrum Analysis: Lensed vs Delensed
print("\nPower spectrum analysis for first test sample...")

# Extract first sample
first_lensed_test = test_lensed_data[0, :, 0]
first_delensed_test = delensed_test[0, :, 0]
first_phi_test = phi_pred_test[0, :, 0]

# Convert to RING ordering for display
first_lensed_ring = hp.reorder(first_lensed_test, n2r=True)
first_delensed_ring = hp.reorder(first_delensed_test, n2r=True)
first_phi_ring = hp.reorder(first_phi_test, n2r=True)

# Plot mollweide projections
fig = plt.figure(figsize=(14, 4))

ax1 = fig.add_subplot(131, projection="mollweide")
plt.axes(ax1)
hp.mollview(
    first_lensed_ring, title="Test Sample: Lensed Map", cmap="viridis", hold=True
)

ax2 = fig.add_subplot(132, projection="mollweide")
plt.axes(ax2)
hp.mollview(
    first_phi_ring, title="U-Net Output: Estimated φ", cmap="viridis", hold=True
)

ax3 = fig.add_subplot(133, projection="mollweide")
plt.axes(ax3)
hp.mollview(
    first_delensed_ring,
    title="Fine-tuned Pipeline: Delensed Map",
    cmap="viridis",
    hold=True,
)

plt.tight_layout()
plt.show()

# Power spectra comparison
print("\nComputing power spectra...")
cl_lensed = hp.anafast(first_lensed_test, lmax=core.lmax, pol=False)
cl_delensed = hp.anafast(first_delensed_test, lmax=core.lmax, pol=False)
ell = np.arange(len(cl_lensed))

plt.figure(figsize=(10, 6))
plt.loglog(ell[2:], cl_lensed[2:], label="Lensed (Input)", linewidth=2.5, color="coral")
plt.loglog(
    ell[2:],
    cl_delensed[2:],
    label="Delensed by Pipeline",
    linewidth=2.5,
    color="steelblue",
)
plt.xlabel(r"$\ell$", fontsize=12)
plt.ylabel(r"$C_\ell$", fontsize=12)
plt.title("Power Spectra: Lensed vs Delensed (First Test Sample)", fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

print("Power spectrum analysis complete!")